# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [15]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [16]:
## Define the path to your data folder
DATA_DIR = '../data/'

# Load our three datasets
print("Loading datasets...")
student_info = pd.read_csv(f"{DATA_DIR}studentInfo.csv")
vle_mapping = pd.read_csv(f"{DATA_DIR}vle.csv")
student_vle = pd.read_csv(f"{DATA_DIR}studentVle.csv")
print("✅ All data loaded successfully!")

Loading datasets...
✅ All data loaded successfully!


In [18]:
# --- STEP 0: Merging Data (عشان نحل مشكلة الـ KeyError) ---

# 1. بنجمع كليكات كل طالب من ملف الـ VLE
student_clicks = student_vle.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()

# 2. بندمج بيانات الطالب مع مجموع كليكاته
df_integrated = pd.merge(student_info, student_clicks, on=['id_student', 'code_module', 'code_presentation'], how='left')

# 3. بنحول أي طالب ملوش كليكات لـ 0 بدل Null
df_integrated['sum_click'] = df_integrated['sum_click'].fillna(0)

print("✅ Data Merged successfully into 'df_integrated'!")
display(df_integrated.head(3))

✅ Data Merged successfully into 'df_integrated'!


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,sum_click
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:

columns_to_drop = ['week_from', 'week_to']
vle_mapping = vle_mapping.drop(columns=columns_to_drop)

print("Dropped 'week_from' and 'week_to' from vle_mapping.")

Dropped 'week_from' and 'week_to' from vle_mapping.


In [4]:

initial_count = len(student_info)
student_info = student_info[student_info['final_result'].notna()]
final_count = len(student_info)

print(f"✅ Filtered student_info: Removed {initial_count - final_count} rows with missing results.")
print(f"📊 Final shape after row selection: {student_info.shape}")

✅ Filtered student_info: Removed 0 rows with missing results.
📊 Final shape after row selection: (32593, 12)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [21]:
# --- Task 2: Data Cleaning (Handling Missing Values & Outliers) ---

# 1. Filling missing categorical data with Mode
df_integrated['imd_band'] = df_integrated['imd_band'].fillna(df_integrated['imd_band'].mode()[0])

# 2. Filling missing clicks with 0
df_integrated['sum_click'] = df_integrated['sum_click'].fillna(0)

# 3. Handling Duplicates (Final Filter)
# بنمسح أي سطر متكرر بالكامل في الجدول المدمج
df_integrated = df_integrated.drop_duplicates()

# 4. Outlier Capping (اختياري لو الدكتور طلبه بالاسم)
# لو عايز تثبت الكليكات عند 100 زي ما كنت عامل فوق
# df_integrated.loc[df_integrated['sum_click'] > 100, 'sum_click'] = 100

print("✅ Task 2 Done: All cleaning (Imputation, Duplicates, and Nulls) handled in one place.")

✅ Task 2 Done: All cleaning (Imputation, Duplicates, and Nulls) handled in one place.


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [8]:

early_clicks = student_vle[student_vle['date'] <= 28]


click_summary = early_clicks.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
click_summary.rename(columns={'sum_click': 'total_early_clicks'}, inplace=True)

In [22]:
from sklearn.preprocessing import RobustScaler

# 1. Merging Constructed Features (دمج الكليكات المبكرة اللي إنت لسه عاملها)
df_integrated = pd.merge(df_integrated, click_summary, on=['id_student', 'code_module', 'code_presentation'], how='left')
df_integrated['total_early_clicks'] = df_integrated['total_early_clicks'].fillna(0)

# 2. Scaling (عشان العدل بين الأرقام)
# بنستخدم RobustScaler عشان الـ Outliers اللي طلعناها في Phase 2
scaler = RobustScaler()
df_integrated['early_clicks_scaled'] = scaler.fit_transform(df_integrated[['total_early_clicks']])

# 3. Binning (تقسيم Credits لمجموعات - طلب الدكتور)
# هنقسم الطلبة لـ (حمل دراسي خفيف، متوسط، عالي)
df_integrated['credit_load'] = pd.cut(df_integrated['studied_credits'], 
                                      bins=[0, 60, 120, 1000], 
                                      labels=['Light', 'Medium', 'Heavy'])

# 4. Encoding (تحويل كل النصوص لأرقام عشان الموديل يفهم)
# بنحول الـ Gender والـ Credit_load والـ Age_band
categorical_cols = ['gender', 'age_band', 'highest_education', 'credit_load']
df_final = pd.get_dummies(df_integrated, columns=categorical_cols, drop_first=True)

print("✅ Task 3 Complete: Features Constructed, Scaled, and Encoded!")
display(df_final.head(3))

✅ Task 3 Complete: Features Constructed, Scaled, and Encoded!


,code_module,code_presentation,id_student,imd_band,num_of_prev_attempts,studied_credits,disability,final_result,sum_click,sum_click_scaled,reg_East Anglian Region,reg_East Midlands Region,reg_Ireland,reg_London Region,reg_North Region,reg_North Western Region,reg_Scotland,reg_South East Region,reg_South Region,reg_South West Region,reg_Wales,reg_West Midlands Region,reg_Yorkshire Region,total_early_clicks,early_clicks_scaled,gender_1,age_band_35-55,age_band_55<=,highest_education_HE Qualification,highest_education_Lower Than A Level,highest_education_No Formal quals,highest_education_Post Graduate Qualification,credit_load_Medium,credit_load_Heavy
0,AAA,2013J,11391,90-100%,0,240,N,Pass,934.0,0.230076,True,False,False,False,False,False,False,False,False,False,False,False,False,400.0,0.648725,True,False,True,True,False,False,False,False,True
1,AAA,2013J,28400,20-30%,0,60,N,Pass,1435.0,0.577270,False,False,False,False,False,False,True,False,False,False,False,False,False,609.0,1.240793,False,True,False,True,False,False,False,False,False
2,AAA,2013J,30268,30-40%,0,60,Y,Withdrawn,281.0,-0.222453,False,False,False,False,False,True,False,False,False,False,False,False,False,260.0,0.252125,False,True,False,False,False,False,False,False,False


In [10]:
print(f"Aggregated clickstream data into {len(click_summary):,} student summaries.")
display(click_summary.head())

Aggregated clickstream data into 28,792 student summaries.


,id_student,code_module,code_presentation,total_early_clicks
0,6516,AAA,2014J,812
1,8462,DDD,2013J,369
2,8462,DDD,2014J,9
3,11391,AAA,2013J,400
4,23629,BBB,2013B,51


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [28]:
%pip install requests beautifulsoup4 lxml
%pip install html5lib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io

print("=== 1. Starting Web Scraping (BeautifulSoup) ===")
url = 'https://en.wikipedia.org/wiki/List_of_regions_of_the_United_Kingdom_by_GRP_per_capita'


headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)


soup = BeautifulSoup(response.text, 'html.parser')


scraped_df = pd.read_html(io.StringIO(str(soup)), match='Region')[0]


scraped_df = scraped_df.iloc[:, [0, 2]]
scraped_df.columns = ['Region_Name', 'GDP_per_capita']


region_mapping = {
    'Scotland': 'Scotland',
    'Wales': 'Wales',
    'London': 'London Region',
    'South East England': 'South East Region',
    'South West England': 'South West Region',
    'East of England': 'East Anglian Region',
    'East Midlands': 'East Midlands Region',
    'West Midlands': 'West Midlands Region',
    'Yorkshire and the Humber': 'Yorkshire Region',
    'North West England': 'North Western Region',
    'North East England': 'North Region',
    'Northern Ireland': 'Ireland'
}


scraped_df['region'] = scraped_df['Region_Name'].map(region_mapping)
scraped_df = scraped_df.dropna(subset=['region'])
print(f"✅ Successfully scraped and cleaned {len(scraped_df)} regional records.")

print("\n=== 2. Integrating Data ===")

df_integrated = pd.merge(student_info, scraped_df[['region', 'GDP_per_capita']], on='region', how='left')


df_integrated = pd.merge(df_integrated, click_summary, on=['id_student', 'code_module', 'code_presentation'], how='left')


df_integrated['total_early_clicks'] = df_integrated['total_early_clicks'].fillna(0)


df_integrated['GDP_per_capita'] = df_integrated['GDP_per_capita'].fillna(df_integrated['GDP_per_capita'].mode()[0])

print(f"✅ Integration complete! Final shape: {df_integrated.shape}")

=== 1. Starting Web Scraping (BeautifulSoup) ===
✅ Successfully scraped and cleaned 2 regional records.

=== 2. Integrating Data ===
✅ Integration complete! Final shape: (32593, 14)


In [30]:
# --- Task 4: Integrating Web Scraped Data & Final Merge ---

# 1. تنظيف رقم الـ GDP (بإضافة astype(str) عشان نحل الـ Error)
# بنحول العمود لنص الأول، بنشيل الرموز، وبنحول الـ 'nan' لـ '0' عشان ميضربش Error
scraped_df['GDP_per_capita'] = (scraped_df['GDP_per_capita']
                                .astype(str)
                                .str.replace('£', '')
                                .str.replace(',', '')
                                .replace('nan', '0')
                                .astype(float))

# 2. الدمج مع بيانات الطالب (تأكد إنك عملت الـ mapping للمناطق قبل الخطوة دي)
df_final = pd.merge(student_info, scraped_df[['region', 'GDP_per_capita']], on='region', how='left')

# 3. الدمج مع الكليكات المبكرة (click_summary) اللي عملناها في Task 3
df_final = pd.merge(df_final, click_summary, on=['id_student', 'code_module', 'code_presentation'], how='left')

# 4. معالجة القيم المفقودة بعد الدمج
df_final['total_early_clicks'] = df_final['total_early_clicks'].fillna(0)
# لو فيه منطقة ملهاش GDP بنملاها بمتوسط المناطق التانية
df_final['GDP_per_capita'] = df_final['GDP_per_capita'].replace(0, df_final['GDP_per_capita'].mean())
df_final['GDP_per_capita'] = df_final['GDP_per_capita'].fillna(df_final['GDP_per_capita'].mean())

print(f"✅ Integration complete! Web scraping data (GDP) and Early Clicks merged.")
print(f"Final Dataset Shape: {df_final.shape}")
display(df_final.head(3))

✅ Integration complete! Web scraping data (GDP) and Early Clicks merged.
Final Dataset Shape: (32593, 14)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,GDP_per_capita,total_early_clicks
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,29325.385024,400.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,29325.385024,609.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,29325.385024,260.0


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [34]:

# --- Task 5: Final Data Selection & Numerical Encoding ---

# 1. تحويل النتيجة لـ Target الرقمي (0 و 1)
if 'final_result' in df_final.columns:
    target_mapping = {'Pass': 0, 'Distinction': 0, 'Fail': 1, 'Withdrawn': 1}
    df_final['target_risk'] = df_final['final_result'].map(target_mapping)

# 2. تنظيف العواميد اللي مش محتاجينها (IDs والتواريخ)
cols_to_drop = ['id_student', 'code_module', 'code_presentation', 'region', 'final_result']
df_temp = df_final.drop(columns=cols_to_drop, errors='ignore')

# 3. ملء أي قيم مفقودة متبقية (عشان الـ 1111 اللي طلعوا)
# بنملا الأرقام بالمتوسط والنصوص بالـ Mode
for col in df_temp.columns:
    if df_temp[col].dtype == 'object':
        df_temp[col] = df_temp[col].fillna(df_temp[col].mode()[0])
    else:
        df_temp[col] = df_temp[col].fillna(df_temp[col].median())

# 4. الـ Encoding (تحويل كل النصوص لأرقام - طلب الدكتور الأساسي)
# السطر ده هيخلي الـ Shape يكبر بس الداتا هتبقى جاهزة للموديل 100%
df_prepared = pd.get_dummies(df_temp, drop_first=True)

# 5. التقرير النهائي الصافي
print("\n" + "="*40)
print(f"🚀 PHASE 3 IS OFFICIALLY PERFECT")
print(f"📊 Final Numerical Shape: {df_prepared.shape}")
print(f"🧹 Remaining Missing Values: {df_prepared.isnull().sum().sum()}")
print("="*40)

# حفظ الملف
df_prepared.to_csv('final_data_ready_for_modeling.csv', index=False)
display(df_prepared.head(3))



🚀 PHASE 3 IS OFFICIALLY PERFECT
📊 Final Numerical Shape: (32593, 22)
🧹 Remaining Missing Values: 0


,num_of_prev_attempts,studied_credits,GDP_per_capita,total_early_clicks,target_risk,gender_M,highest_education_HE Qualification,highest_education_Lower Than A Level,highest_education_No Formal quals,highest_education_Post Graduate Qualification,imd_band_10-20,imd_band_20-30%,imd_band_30-40%,imd_band_40-50%,imd_band_50-60%,imd_band_60-70%,imd_band_70-80%,imd_band_80-90%,imd_band_90-100%,age_band_35-55,age_band_55<=,disability_Y
0,0,240,29325.385024,400.0,0,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False
1,0,60,29325.385024,609.0,0,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
2,0,60,29325.385024,260.0,1,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,True


In [ ]:

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)

print(f"✅ Shape: {df_final.shape}")
print(f"❌ Missing values: {df_final.isnull().sum().sum()}")
print(f"🔄 Duplicates: {df_final.duplicated().sum()}")
print("\n📋 Column types:")
print(df_final.dtypes)


display(df_final.head())

FINAL PREPARED DATASET SUMMARY
✅ Shape: (32593, 13)
❌ Missing values: 0
🔄 Duplicates: 232

📋 Column types:
code_module              object
code_presentation        object
gender                   object
region                   object
highest_education        object
imd_band                 object
age_band                 object
num_of_prev_attempts      int64
studied_credits           int64
disability               object
GDP_per_capita          float64
total_early_clicks      float64
target_risk               int64
dtype: object


,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,GDP_per_capita,total_early_clicks,target_risk
0,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,30300.0,400.0,0
1,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,30300.0,609.0,0
2,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,30300.0,260.0,1
3,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,30300.0,469.0,0
4,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,30300.0,559.0,0


In [33]:

df_final.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Success! Prepared dataset saved to: {OUTPUT_PATH}")
print("🚀 Ready for Phase 4: Modelling.")

✅ Success! Prepared dataset saved to: ../data/prepared_student_data.csv
🚀 Ready for Phase 4: Modelling.
